In [ ]:
import logging
import time
import csv
import re # filename cleaning
from datetime import datetime
from googleapiclient.discovery import build # YouTube API
from googleapiclient.errors import HttpError # API Error Handling

In [ ]:
API_KEY = "" # DELETED FOR SECURITY REASONS, PLEASE INSERT YOUR OWN API KEY HERE!

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class YouTubeCommentScraper:
    def __init__(self, api_key):
        self.youtube = build('youtube', 'v3', developerKey=api_key)
        self.last_request_time = 0
        self.min_interval = 1.0
        self.video_title_cache = {}

    def _rate_limit(self):
        current_time = time.time()
        elapsed = current_time - self.last_request_time
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self.last_request_time = time.time()

    def get_video_title(self, video_id):
        if video_id in self.video_title_cache:
            return self.video_title_cache[video_id]
        
        self._rate_limit()
        try:
            request = self.youtube.videos().list(
                part='snippet',
                id=video_id
            )
            response = request.execute()
            
            if response['items']:
                title = response['items'][0]['snippet']['title']
                # Clean invalid chars, limit length
                title_clean = re.sub(r'[<>:"/\\|?*]', '-', title)[:100].strip()
                title_clean = re.sub(r'[- ]+', ' ', title_clean).strip()
                self.video_title_cache[video_id] = title_clean
                logger.info(f"Fetched title for {video_id}: {title_clean}")
                return title_clean
            else:
                title = f"VIDEO_{video_id}"
                self.video_title_cache[video_id] = title
                return title
        except HttpError as e:
            logger.error(f"Error fetching title for {video_id}: {e}")
            return f"VIDEO_{video_id}"

    def get_video_comments(self, video_id, max_results=10000):
        video_title = self.get_video_title(video_id)  # video_title debug
        comments = []
        next_page_token = None

        logger.info(f"Target: {max_results} comments for video {video_id} ({video_title})")
        while len(comments) < max_results:
            self._rate_limit()
            try:
                request = self.youtube.commentThreads().list(
                    part='snippet',
                    videoId=video_id,
                    maxResults=100,
                    pageToken=next_page_token,
                    textFormat='plainText',
                    order='time'
                )
                response = request.execute()

                for item in response['items']:
                    snippet = item['snippet']['topLevelComment']['snippet']
                    
                    comment = {
                        'comment_id': item['id'],
                        'video_id': video_id,
                        'video_title': video_title, 
                        'text': snippet['textOriginal'].strip(),
                        'like_count': snippet.get('likeCount', 0),
                        'published_at': snippet.get('publishedAt', ''),
                        'is_reply': False
                    }
                    comments.append(comment)

                next_page_token = response.get('nextPageToken')
                logger.info(f"Collected {len(comments)} / {max_results} for video {video_id}")

                if not next_page_token:
                    break

            except HttpError as e:
                logger.error(f"API Error for video {video_id}: {e}")
                break

        return comments[:max_results]

In [ ]:
def main():
    video_ids = [ "kBdfcR-8hEY", "Unzc731iCUY", "NNnIGh9g6fA", "I3GWzXRectE", "rfscVS0vtbw"] # contoh video IDs
    
    scraper = YouTubeCommentScraper(API_KEY)
    
    for video_id in video_ids:
        logger.info(f"Starting collection for video: {video_id}")
        comments = scraper.get_video_comments(video_id, max_results=10000)
        
        video_title = scraper.get_video_title(video_id)  
        filename = f"{video_title}.csv"  
        
        with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=['comment_id', 'video_id', 'video_title', 'text', 'like_count', 'published_at', 'is_reply'])
            writer.writeheader()
            writer.writerows(comments)

        logger.info(f"Comments saved to {filename} ({len(comments)} entries)")
        time.sleep(1)

if __name__ == "__main__":
    main()